# Lab 7 — Deep Learning, from One Neuron to a Solved Growth Model

**ECON 282E · Session 7 · November 5, 2026**

Four blocks and three appendices, in the order of the decks.

| Block | You will | Deck |
|---|---|---|
| **A** | use tensors and autograd, check backpropagation by hand, and test the training loop where the answer is known | 7.A3, 7.B1 |
| **B** | fit a curve and watch it fail outside the data; place kinks by hand and by training; see which exact fit gradient descent picks | 7.A2, 7.B1, 7.B4 |
| **C** | forecast next month's 10-year Treasury yield against a random walk | 7.B1 |
| **D** | solve the growth model by supervised learning, the Euler equation supplying the labels | 7.B2 |
| Appendices | leakage in a text classifier; one input through four architectures; what a gradient costs | 7.A3, 7.B4 |

The notebook runs offline on a CPU from a clean kernel in about a minute. The data are frozen files in
`tools/figures/data/s07/`; every reported number is printed by the cell that produces it, and the cell
compares it with the number on the slide.

> The code below the next cell is **the code the decks print and the ledger runs**, inlined from
> `slides/S07/code/`. Where your run disagrees with a slide, the notebook is the check on the deck.

In [ ]:
%matplotlib inline
import json, math, sys, time, hashlib
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

ROOT = Path.cwd().parent if Path.cwd().name == "labs" else Path.cwd()
DATA = ROOT / "tools" / "figures" / "data" / "s07"
assert (DATA / "yield_panel.csv").exists(), f"run the notebook from the course folder or labs/: {DATA}"
torch.set_num_threads(1)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("torch", torch.__version__, "| numpy", np.__version__, "| pandas", pd.__version__, "| device: cpu")

def close(a, b, rel=1e-3, abs_=1e-9):
    return abs(a - b) <= max(abs_, rel * abs(b))

In [ ]:
LEDGER = {'backprop_sigmoid_dL_dw1': 0.06334785,
 'backprop_sigmoid_dL_dw2': 0.0838954,
 'backprop_relu_dL_dw1': -7.776,
 'backprop_relu_loss_new': 1.7488,
 'version0_ols': [1.0073, 1.9972, -0.5023],
 'curve_mse': (0.0231, 0.0229),
 'curve_net_at_30': 13.52,
 'kinks_fixed_mse': {4: 0.0058935696, 8: 0.0003077511},
 'kinks_learned_mse': {2: 0.165067777, 32: 0.0001772264, 4: 0.005427693, 8: 0.0006302951},
 'adaptivity_ratio': 396.2,
 'minnorm': (0.956, 2.94),
 'norm_peak': (7, 8.311),
 'norm_vary_n': (0.616, 1.538),
 'overfit_clean_noisy_at_n': (0.4473, 0.6053),
 'yield_rmse': {'ar1': 0.2647, 'mlp': 0.3078, 'ols': 0.2855, 'random_walk': 0.2658},
 'yield_ensemble_rmse': 0.2956,
 'yield_width': 4,
 'ti_iterations': 292,
 'ti_log10_max_euler': -3.445,
 'held_out_log10_max_euler': {'0': -1.94952, '1': -3.3406, '2': -2.15377},
 'off_domain_log10_max_euler': {'0': -1.29215, '1': -1.59412, '2': -1.42146},
 'held_out_max_policy_error': {'0': 0.00641, '1': 0.00041, '2': 0.00564},
 'k_ss_relative_gap': {'0': 0.00467, '1': 0.0018, '2': 2e-05},
 'leakage': (0.97, 0.5)}

### The code the decks print

Three files, run unchanged by `tools/figures/s07_lab_generate.py`: the small networks of 7.A1--7.B4
(`s07_nets.py`), the yield forecast (`s07_yield.py`) and the growth ladder (`s07_growth.py`). Read the
first one once now; the rest of the notebook calls it.

In [ ]:
# ---- s07_nets.py, inlined from slides/S07/code -- edit it there, not here ----
from __future__ import annotations

import math

import numpy as np
import torch
from torch import nn


# ------------------------------------------------------------------ 7.B1: tensors and shapes
def broadcasting_demo() -> dict[str, object]:
    k = torch.linspace(0.5, 2.0, 4)      # shape (4,)
    K = k.reshape(-1, 1)                 # shape (4, 1): four observations, one feature
    print(k.shape, K.shape, K.dtype)     # torch.Size([4]) torch.Size([4, 1]) torch.float32
    print((K - k).shape)                 # torch.Size([4, 4])   <- not what you meant
    return {"k_shape": list(k.shape), "K_shape": list(K.shape), "dtype": str(K.dtype),
            "broadcast_shape": list((K - k).shape)}


# ------------------------------------------------------------------ 7.A3: backprop by hand
def backprop_sigmoid() -> dict[str, float]:
    """One sigmoid hidden unit, one sigmoid output, L = 0.5 (yhat - t)^2; checked by autograd."""
    sig = lambda z: 1.0 / (1.0 + math.exp(-z))
    x, t, w1, b1, w2, b2, eta = 1.0, 0.0, 0.5, 0.0, 2.0, 0.0, 0.1
    z1 = w1 * x + b1; h = sig(z1); z2 = w2 * h + b2; yhat = sig(z2); L = 0.5 * (yhat - t) ** 2
    d2 = (yhat - t) * yhat * (1 - yhat)          # dL/dz2
    dw2, dh = d2 * h, d2 * w2
    d1 = dh * h * (1 - h)                         # dL/dz1
    dw1 = d1 * x
    p = [torch.tensor(v, requires_grad=True, dtype=torch.float64) for v in (w1, b1, w2, b2)]
    loss = 0.5 * (torch.sigmoid(p[2] * torch.sigmoid(p[0] * x + p[1]) + p[3]) - t) ** 2
    loss.backward()
    return {"z1": z1, "h": h, "z2": z2, "yhat": yhat, "loss": L, "delta2": d2, "dL_dw2": dw2,
            "dL_db2": d2, "dL_dh": dh, "dyhat_dz2": yhat * (1 - yhat), "dh_dz1": h * (1 - h),
            "delta1": d1, "dL_dw1": dw1,
            "w2_new": w2 - eta * dw2, "w1_new": w1 - eta * dw1, "eta": eta,
            "autograd_dL_dw1": p[0].grad.item(), "autograd_dL_dw2": p[2].grad.item()}


def backprop_relu() -> dict[str, float]:
    """The four-parameter ReLU example 7.A3 works by hand and 7.B1 checks with backward()."""
    x, y = 2.0, 3.0
    w1, b1, v, c = (torch.tensor(a, requires_grad=True) for a in (0.5, -0.1, 1.2, 0.3))
    z = w1 * x + b1; h = torch.relu(z); yhat = v * h + c; L = (yhat - y) ** 2
    L.backward()
    eta = 0.01
    new = [a.item() - eta * a.grad.item() for a in (w1, b1, v, c)]
    z_n = new[0] * x + new[1]
    L_new = (new[2] * max(z_n, 0.0) + new[3] - y) ** 2
    # a second hidden unit on the inactive side receives exactly zero gradient
    w_dead = torch.tensor(-0.5, requires_grad=True)
    L2 = (v.detach() * torch.relu(w1.detach() * x + b1.detach())
          + 0.7 * torch.relu(w_dead * x + 0.2) + c.detach() - y) ** 2
    L2.backward()
    return {"z": z.item(), "h": h.item(), "yhat": yhat.item(), "loss": L.item(),
            "dL_dyhat": 2 * (yhat.item() - y), "dL_dh": 2 * (yhat.item() - y) * v.item(),
            "dL_dv": v.grad.item(), "dL_dc": c.grad.item(), "dL_dw1": w1.grad.item(),
            "dL_db1": b1.grad.item(), "eta": eta, "w1_new": new[0], "v_new": new[2],
            "loss_new": L_new, "dead_z": -0.5 * x + 0.2, "dead_grad": w_dead.grad.item()}


# ------------------------------------------------------------------ 7.B1: the loop at version 0
def version0() -> dict[str, object]:
    torch.manual_seed(0)
    X = torch.randn(200, 2)
    y = 1.0 + X @ torch.tensor([[2.0], [-0.5]]) + 0.1 * torch.randn(200, 1)
    model = nn.Linear(2, 1)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    for step in range(300):
        prediction = model(X)                    # 1 forward
        loss = ((prediction - y) ** 2).mean()    # 2 loss
        optimizer.zero_grad()                    # 3 clear the old gradients
        loss.backward()                          # 4 backward
        optimizer.step()                         # 5 update
    ols = np.linalg.lstsq(np.column_stack([np.ones(200), X.numpy()]), y.numpy(),
                          rcond=None)[0].ravel()
    learned = np.r_[model.bias.item(), model.weight.detach().numpy().ravel()]
    return {"ols": ols.tolist(), "learned": learned.tolist(),
            "max_gap": float(np.max(np.abs(ols - learned)))}


# ------------------------------------------------------------------ 7.B1: a network is a class
class SimpleNet(nn.Module):
    def __init__(self, hidden=32, act=nn.ReLU):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, hidden), act(), nn.Linear(hidden, 1))
    def forward(self, x):
        return self.net(x)


def one_training_step(loss_fn, X_tr, y_tr, X_va, y_va, weight_decay=1e-3):
    """The optimizer frame of 7.B1, as one runnable step on a one-input SimpleNet."""
    torch.manual_seed(1); net = SimpleNet()              # same seed, same initial draw
    optimizer = torch.optim.Adam(net.parameters(), lr=3e-3, weight_decay=weight_decay)
    optimizer.zero_grad()                                # 3 clear the old gradients
    loss = loss_fn(net(X_tr), y_tr); loss.backward(); optimizer.step()
    net.eval()                                           # evaluation mode
    with torch.no_grad():                                # no tape: nothing to differentiate
        val = float(loss_fn(net(X_va), y_va))
    return float(loss.detach()), val


def seed_check() -> dict[str, object]:
    torch.manual_seed(1); a = SimpleNet()
    torch.manual_seed(1); b = SimpleNet()
    return {"parameters": sum(p.numel() for p in a.parameters()),
            "same_initial_weights": bool(torch.equal(a.net[0].weight, b.net[0].weight))}


# ------------------------------------------------------------------ 7.B1: a curve whose answer is known
def curve_data():
    torch.manual_seed(42)
    w = torch.rand(400, 1) * 10 + 1
    c = torch.log(w) + 0.2 * w + 0.15 * torch.randn(400, 1)
    idx = torch.randperm(400); tr, te = idx[:300], idx[300:]
    mu, sd = w[tr].mean(), w[tr].std()
    return w, c, tr, te, mu, sd


def fit_curve_on(w, c, tr, mu, sd, hidden=32, act=nn.ReLU, epochs=1500, seed=0):
    torch.manual_seed(seed)
    net = SimpleNet(hidden, act)
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    for _ in range(epochs):
        loss = ((net((w[tr] - mu) / sd) - c[tr]) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    net.eval()
    return net


def score_curve(net, w, c, tr, te, mu, sd) -> dict[str, float]:
    with torch.no_grad():
        mse = {name: ((net((w[i] - mu) / sd) - c[i]) ** 2).mean().item()
               for name, i in (("train", tr), ("test", te))}
    return mse


# ------------------------------------------------------------------ 7.A2: kinks
SIN_X = np.linspace(-math.pi, math.pi, 121)


def kinks_fixed(H: int, x: np.ndarray = SIN_X, target=np.sin):
    """Evenly spaced kinks, coefficients by least squares: no network yet."""
    knots = np.linspace(x[0], x[-1], H, endpoint=False)
    Phi = np.column_stack([np.ones_like(x)] + [np.maximum(0.0, x - s) for s in knots])
    coef = np.linalg.lstsq(Phi, target(x), rcond=None)[0]
    fit = Phi @ coef
    return fit, knots, float(np.mean((fit - target(x)) ** 2))


def kinks_learned(H: int, x: np.ndarray = SIN_X, target=np.sin, steps: int = 3000, seed: int = 1):
    """Train the kinks with the coefficients: a one-hidden-layer ReLU network, Adam."""
    if x is SIN_X and target is np.sin:          # the sin x sample is built in torch, as drawn
        xs = torch.linspace(-math.pi, math.pi, len(x)).reshape(-1, 1)
        ys = torch.sin(xs)
    else:
        xs = torch.tensor(x, dtype=torch.float32).reshape(-1, 1)
        ys = torch.tensor(target(x), dtype=torch.float32).reshape(-1, 1)
    torch.manual_seed(seed)
    m = SimpleNet(H)
    opt = torch.optim.Adam(m.parameters(), lr=1e-2)
    for _ in range(steps):
        loss = ((m(xs) - ys) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        fit = m(xs).squeeze().numpy()
        kinks = (-m.net[0].bias / m.net[0].weight.squeeze()).numpy()
    kinks = np.sort(kinks[(kinks > x[0]) & (kinks < x[-1])])
    return fit, kinks, float(loss.detach())


# ------------------------------------------------------------------ 7.B4: the minimum-norm fit
def hinge_features(x: np.ndarray, shifts: np.ndarray) -> np.ndarray:
    return np.column_stack([np.ones_like(x)] + [np.maximum(0.0, x - s) for s in shifts])


def minimum_norm_example():
    """Seven points of sin x, 41 fixed hinge features: the smallest exact fit and another one."""
    xd = np.linspace(-np.pi, np.pi, 7); yd = np.sin(xd)
    shifts = np.linspace(-np.pi, np.pi, 40)
    Phi = hinge_features(xd, shifts)
    a_min = np.linalg.pinv(Phi) @ yd
    null = np.linalg.svd(Phi)[2][7:].T
    a_other = a_min + null @ np.random.default_rng(0).normal(size=null.shape[1]) * 0.6
    return xd, yd, shifts, a_min, a_other


def van_der_corput(k: int, base: int = 2) -> float:
    f, o = 1.0, 0.0
    while k > 0:
        f /= base; o += f * (k % base); k //= base
    return o


POOL = np.array([-np.pi + 2 * np.pi * van_der_corput(k + 1) for k in range(300)])      # nested hinges
XPOOL = np.array([-np.pi + 2 * np.pi * van_der_corput(k + 1, 3) for k in range(300)])  # nested data
X_TEST = np.linspace(-np.pi, np.pi, 401)


def nested_features(x: np.ndarray, p: int) -> np.ndarray:
    return np.column_stack([np.ones_like(x)] + [np.maximum(0.0, x - s) for s in POOL[:p - 1]])


def min_norm_solve(n: int, p: int, nested_x: bool = False):
    """Minimum-norm fit with n points and p nested features: (norm, test RMSE, max train error, dof)."""
    xd = XPOOL[:n] if nested_x else np.linspace(-np.pi, np.pi, n)
    yd = np.sin(xd)
    Phi = nested_features(xd, p)
    a = np.linalg.pinv(Phi) @ yd
    return (float(np.linalg.norm(a)),
            float(np.sqrt(np.mean((nested_features(X_TEST, p) @ a - np.sin(X_TEST)) ** 2))),
            float(np.abs(Phi @ a - yd).max()),
            int(p - np.linalg.matrix_rank(Phi)))


P_SWEEP = sorted(set(list(range(2, 61)) + list(range(62, 257, 3)) + [256]))


def overfit_vs_selection(n: int = 7, sigma: float = 0.30, draws: int = 200):
    """Test RMSE of the minimum-norm fit, with y = sin x exactly and with noise, at every p."""
    xd = np.linspace(-np.pi, np.pi, n); clean = np.sin(xd)
    rng = np.random.default_rng(0)
    ys = [clean + sigma * rng.normal(size=n) for _ in range(draws)]
    rows = []
    for p in P_SWEEP:
        Phi, Pt = nested_features(xd, p), nested_features(X_TEST, p)
        pinv = np.linalg.pinv(Phi)
        a = pinv @ clean
        te0 = float(np.sqrt(np.mean((Pt @ a - np.sin(X_TEST)) ** 2)))
        tr0 = float(np.sqrt(np.mean((Phi @ a - clean) ** 2)))
        tes, trs = [], []
        for y in ys:
            b = pinv @ y
            tes.append(np.mean((Pt @ b - np.sin(X_TEST)) ** 2)); trs.append(np.mean((Phi @ b - y) ** 2))
        rows.append((p, te0, float(np.sqrt(np.mean(tes))), tr0, float(np.sqrt(np.mean(trs)))))
    return rows


# ------------------------------------------------------------------ 7.B4: one input, four layers
def four_architectures() -> dict[str, object]:
    """Hand-chosen weights, so every number can be checked by hand; asserted."""
    x = np.array([1.0, 2.0, 3.0, 4.0])
    w_dense = np.array([0.5, -0.5, 0.5, -0.5])
    y_dense = float(w_dense @ x)
    filt = np.array([1.0, -1.0])
    W_conv = np.zeros((3, 4))
    for r in range(3):
        W_conv[r, r:r + 2] = filt
    y_conv = W_conv @ x
    h, hs = 0.0, []
    for t in range(4):
        h = 0.5 * h + 1.0 * x[t]
        hs.append(h)
    nbrs = {0: [1], 1: [0, 2], 2: [1, 3], 3: [2]}
    gnn = lambda v: np.array([v[i] + 0.5 * np.mean([v[u] for u in nbrs[i]]) for i in range(4)])
    y_gnn = gnn(x)
    perm = [3, 2, 1, 0]
    y_gnn_p = gnn(x[perm])
    sh = [2, 0, 3, 1]
    assert y_dense == -1.0 and np.array_equal(y_conv, [-1.0, -1.0, -1.0])
    assert np.allclose(hs, [1.0, 2.5, 4.25, 6.125]) and np.allclose(y_gnn, [2.0, 3.0, 4.5, 5.5])
    assert np.allclose(y_gnn_p, y_gnn[perm]) and np.isclose(y_gnn_p.sum(), y_gnn.sum())
    return {"x": x.tolist(), "dense": {"w": w_dense.tolist(), "y": y_dense, "params": 5},
            "conv": {"filter": filt.tolist(), "W": W_conv.tolist(), "y": y_conv.tolist(), "params": 3},
            "rnn": {"w_h": 0.5, "w_x": 1.0, "h": hs, "params": 3},
            "gnn": {"w_self": 1.0, "w_nbr": 0.5, "h": y_gnn.tolist(), "permuted": y_gnn_p.tolist(),
                    "readout_sum": float(y_gnn.sum()), "params": 3},
            "input_shuffle": {"order": sh, "dense": float(w_dense[sh] @ x[sh]),
                              "conv": (W_conv @ x[sh]).tolist()}}


# ------------------------------------------------------------------ 7.A3: leakage, in a text classifier
SUBJECTS = ["The committee", "Federal Reserve officials", "Policymakers", "Several members",
            "The central bank", "The latest statement"]
HAWKISH = ["signaled further interest rate hikes", "favored a more restrictive policy stance",
           "stressed that inflation remains too high", "supported raising rates to curb inflation",
           "warned about the risk of overheating", "leaned toward additional monetary tightening",
           "emphasized keeping rates higher for longer", "pointed to strong demand and rising prices"]
DOVISH = ["signaled possible interest rate cuts", "favored a more accommodative policy stance",
          "stressed that economic growth is slowing", "supported lowering rates to support activity",
          "warned about the risk of a downturn", "leaned toward monetary easing",
          "emphasized supporting employment and jobs", "pointed to weak demand and cooling prices"]


def text_leakage() -> dict[str, float]:
    """Sentences are subject x phrase: a random split tests recall of a template already seen."""
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.linear_model import LogisticRegression
    rows = ([(f"{s} {p}.", 1, i) for s in SUBJECTS for i, p in enumerate(HAWKISH)]
            + [(f"{s} {p}.", 0, i) for s in SUBJECTS for i, p in enumerate(DOVISH)])
    text, label, phrase = map(np.array, zip(*rows))

    def accuracy(train_mask):
        vec = CountVectorizer(stop_words="english").fit(text[train_mask])
        clf = LogisticRegression(max_iter=1000).fit(vec.transform(text[train_mask]), label[train_mask])
        return float((clf.predict(vec.transform(text[~train_mask])) == label[~train_mask]).mean())

    rng = np.random.default_rng(0)
    return {"sentences": len(text), "random_split": accuracy(rng.random(len(text)) < 0.7),
            "by_phrase": accuracy(phrase < 6)}

In [ ]:
# ---- s07_yield.py, inlined from slides/S07/code -- edit it there, not here ----
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

RAW = DATA / "raw"
PANEL = DATA / "yield_panel.csv"
MANIFEST = DATA / "yield_panel_manifest.json"
BASELINE = DATA / "baseline_predictions.csv"

SERIES = ("DGS10", "DGS3MO", "FEDFUNDS", "UNRATE", "CPIAUCSL")
FEATURES = ["y10", "spread", "fedfunds_lag1", "unrate_lag1", "inflation_lag1"]
TRAIN = ("1990-01-31", "2014-12-31")
VALID = ("2015-01-31", "2019-12-31")
TEST_START = "2020-01-31"


def file_sha256(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def _read(series: str) -> pd.Series:
    frame = pd.read_csv(RAW / f"{series}.csv", parse_dates=["observation_date"], na_values=["."])
    return frame.set_index("observation_date")[series].astype(float).dropna()


def build_panel(download_date: str, last_complete_month: str) -> pd.DataFrame:
    """Assemble the monthly panel from the frozen raw files and write it with its manifest."""
    daily10, daily3m = _read("DGS10"), _read("DGS3MO")
    def last_of_month(daily: pd.Series) -> pd.Series:
        out = daily.groupby(daily.index.to_period("M")).last()
        out.index = out.index.to_timestamp("M").normalize()
        return out

    month_end = pd.DataFrame({"y10": last_of_month(daily10), "y3m": last_of_month(daily3m)})
    monthly = pd.DataFrame({s.lower(): _read(s) for s in ("FEDFUNDS", "UNRATE", "CPIAUCSL")})
    monthly.index = monthly.index.to_period("M").to_timestamp("M").normalize()  # stamp at month end
    monthly["inflation"] = 100.0 * (monthly["cpiaucsl"] / monthly["cpiaucsl"].shift(12) - 1.0)

    panel = month_end.join(monthly[["fedfunds", "unrate", "inflation"]], how="left")
    panel["spread"] = panel["y10"] - panel["y3m"]
    for name in ("fedfunds", "unrate", "inflation"):
        panel[f"{name}_lag1"] = panel[name].shift(1)       # month t-1 value, known at end of t
    panel["target_date"] = panel.index.to_series().shift(-1)
    panel["y10_next"] = panel["y10"].shift(-1)              # the label: month-end yield at t+1
    panel = panel[["target_date", "y10_next"] + FEATURES]
    panel = panel[panel["target_date"] <= last_complete_month].dropna()
    panel = panel[panel["target_date"] >= TRAIN[0]]
    panel.index.name = "date"
    panel.to_csv(PANEL, float_format="%.6f")
    MANIFEST.write_text(json.dumps({
        "source": "FRED, https://fred.stlouisfed.org (fredgraph.csv endpoint)",
        "download_date": download_date,
        "vintage": "current vintage, not real time",
        "last_complete_month": last_complete_month,
        "raw_sha256": {s: file_sha256(RAW / f"{s}.csv") for s in SERIES},
        "panel_sha256": file_sha256(PANEL),
        "rows": int(len(panel)),
    }, indent=2), encoding="utf-8")
    return panel


def load_panel() -> pd.DataFrame:
    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
    if file_sha256(PANEL) != manifest["panel_sha256"]:
        raise RuntimeError("yield_panel.csv does not match its manifest")
    return pd.read_csv(PANEL, index_col="date", parse_dates=["date", "target_date"])


def split(panel: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Split by the date of the label; the DatetimeIndex is kept."""
    t = panel["target_date"]
    return {
        "train": panel[(t >= TRAIN[0]) & (t <= TRAIN[1])],
        "valid": panel[(t >= VALID[0]) & (t <= VALID[1])],
        "test": panel[t >= TEST_START],
    }


def metrics(actual: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    err = np.asarray(actual) - np.asarray(predicted)
    return {"rmse": float(np.sqrt(np.mean(err**2))), "mae": float(np.mean(np.abs(err)))}


# ---------------------------------------------------------------- the three baselines

def random_walk(frame: pd.DataFrame) -> np.ndarray:
    return frame["y10"].to_numpy()


def fit_ar1(train: pd.DataFrame):
    X = np.column_stack([np.ones(len(train)), train["y10"]])
    coef, *_ = np.linalg.lstsq(X, train["y10_next"].to_numpy(), rcond=None)
    return lambda frame: coef[0] + coef[1] * frame["y10"].to_numpy()


def fit_ols(train: pd.DataFrame):
    X = np.column_stack([np.ones(len(train)), train[FEATURES]])
    coef, *_ = np.linalg.lstsq(X, train["y10_next"].to_numpy(), rcond=None)
    return lambda frame: np.column_stack([np.ones(len(frame)), frame[FEATURES]]) @ coef


# ---------------------------------------------------------------- the network

class Scaler:
    """Mean and standard deviation from the training window only."""

    def __init__(self, train: pd.DataFrame) -> None:
        self.mean = train[FEATURES].mean()
        self.std = train[FEATURES].std()

    def __call__(self, frame: pd.DataFrame) -> torch.Tensor:
        return torch.tensor(((frame[FEATURES] - self.mean) / self.std).to_numpy(), dtype=torch.float32)


def change(frame: pd.DataFrame) -> torch.Tensor:
    """The network predicts the change, so a network that outputs zero is the random walk."""
    return torch.tensor((frame["y10_next"] - frame["y10"]).to_numpy(), dtype=torch.float32).reshape(-1, 1)


def fit_mlp(parts: dict[str, pd.DataFrame], width: int, seed: int,
            max_epochs: int = 400, patience: int = 40, weight_decay: float = 1e-3):
    """Full-batch Adam; early stopping on the validation window; returns predictor and the curve."""
    torch.manual_seed(seed)
    torch.set_num_threads(1)
    scaler = Scaler(parts["train"])
    X_tr, y_tr = scaler(parts["train"]), change(parts["train"])
    X_va, y_va = scaler(parts["valid"]), change(parts["valid"])
    net = nn.Sequential(nn.Linear(len(FEATURES), width), nn.ReLU(), nn.Linear(width, 1))
    optimizer = torch.optim.Adam(net.parameters(), lr=3e-3, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    best, best_state, since, curve = float("inf"), None, 0, []
    for epoch in range(max_epochs):
        net.train()
        loss = loss_fn(net(X_tr), y_tr)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        net.eval()
        with torch.no_grad():
            val = float(loss_fn(net(X_va), y_va))
        curve.append((epoch, float(loss.detach()), val))
        if val < best - 1e-6:
            best, since = val, 0
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            since += 1
            if since >= patience:
                break
    net.load_state_dict(best_state)
    net.eval()

    def predict(frame: pd.DataFrame) -> np.ndarray:
        with torch.no_grad():
            return frame["y10"].to_numpy() + net(scaler(frame)).squeeze(1).numpy()

    return predict, curve, best


def choose_width(parts: dict[str, pd.DataFrame], widths=(4, 8, 16), seed: int = 0) -> tuple[int, dict[int, float]]:
    """Width is chosen on the validation window and nowhere else."""
    scores = {w: fit_mlp(parts, w, seed)[2] for w in widths}
    return min(scores, key=scores.get), scores


def run_all(seeds=range(10)) -> dict[str, object]:
    panel = load_panel()
    parts = split(panel)
    test = parts["test"]
    actual = test["y10_next"].to_numpy()
    preds = {"random_walk": random_walk(test), "ar1": fit_ar1(parts["train"])(test),
             "ols": fit_ols(parts["train"])(test)}
    width, width_scores = choose_width(parts)
    mlp_runs = [fit_mlp(parts, width, s)[0](test) for s in seeds]
    table = {name: metrics(actual, p) for name, p in preds.items()}
    rmses = [metrics(actual, p)["rmse"] for p in mlp_runs]
    maes = [metrics(actual, p)["mae"] for p in mlp_runs]
    table["mlp"] = {"rmse": float(np.mean(rmses)), "mae": float(np.mean(maes)),
                    "rmse_min": float(np.min(rmses)), "rmse_max": float(np.max(rmses)),
                    "width": width, "seeds": len(rmses)}
    preds["mlp_seed_mean"] = np.mean(mlp_runs, axis=0)
    support = {c: (float(parts["train"][c].min()), float(parts["train"][c].max()),
                   float(((test[c] < parts["train"][c].min()) | (test[c] > parts["train"][c].max())).mean()))
               for c in FEATURES}
    return {"parts": parts, "actual": actual, "predictions": preds, "table": table,
            "width_scores": width_scores, "support": support}


def freeze_baseline(result: dict[str, object]) -> pd.DataFrame:
    """The Session 9 interface: dated forecasts a text signal dated <= t can be scored against."""
    test = result["parts"]["test"]
    out = pd.DataFrame({"forecast_made": test.index, "target_date": test["target_date"].to_numpy(),
                        "actual": result["actual"]})
    for name, p in result["predictions"].items():
        out[name] = p
    out.to_csv(BASELINE, index=False, float_format="%.6f")
    return out


if __name__ == "__main__":
    import sys
    if len(sys.argv) == 3:                       # python session02_yield.py 2026-09-20 2026-08-31
        print(build_panel(sys.argv[1], sys.argv[2]).tail(3))
    result = run_all()
    freeze_baseline(result)
    print(json.dumps(result["table"], indent=2))
    print("validation MSE by width:", result["width_scores"])
    print("share of test months outside the training range:", {k: round(v[2], 2) for k, v in result["support"].items()})
    print({k: len(v) for k, v in result["parts"].items()})

In [ ]:
# ---- s07_growth.py, inlined from slides/S07/code -- edit it there, not here ----
from __future__ import annotations

import hashlib
import json
import math
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import torch
from torch import nn



def file_sha256(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


@dataclass(frozen=True)
class Rung:
    name: str
    alpha: float
    beta: float
    delta: float
    gamma: float
    productivity: float = 1.0
    rho: float = 0.0
    sigma: float = 0.0  # st. dev. of the log-TFP innovation; 0 means deterministic
    steps: int = 500
    learning_rate: float = 3e-3
    batch_size: int = 256
    hidden: int = 32
    seed: int = 2026
    informed_start: bool = False  # start the saving share at its steady-state value

    @property
    def stochastic(self) -> bool:
        return self.sigma > 0.0

    @property
    def k_ss(self) -> float:
        """Deterministic steady state (z = 1)."""
        rental = 1.0 / self.beta - 1.0 + self.delta
        return (self.alpha * self.productivity / rental) ** (1.0 / (1.0 - self.alpha))

    @property
    def log_z_bound(self) -> float:
        """Three unconditional standard deviations of log z."""
        if not self.stochastic:
            return 0.0
        return 3.0 * self.sigma / math.sqrt(1.0 - self.rho**2)


RUNGS = {
    0: Rung("rung 0: log, delta=1, deterministic", 0.33, 0.96, 1.0, 1.0),
    1: Rung("rung 1: CRRA gamma=2, delta=0.025, deterministic", 0.33, 0.99, 0.025, 2.0,
            steps=3000, learning_rate=2e-3, informed_start=True),
    2: Rung("rung 2: Brock-Mirman, log, delta=1, AR(1) TFP", 0.33, 0.96, 1.0, 1.0,
            rho=0.9, sigma=0.05, steps=1500),
}

DOMAIN = (0.2, 2.0)       # training band, as multiples of k_ss
OFF_DOMAIN = (0.1, 3.0)   # reporting band outside the training support
GH_NODES = 7


def gauss_hermite(n: int = GH_NODES) -> tuple[torch.Tensor, torch.Tensor]:
    """Nodes and weights for E[h(eps)], eps ~ N(0,1): sum_j w_j h(x_j)."""
    x, w = np.polynomial.hermite.hermgauss(n)
    nodes = torch.tensor(math.sqrt(2.0) * x, dtype=torch.float32)
    weights = torch.tensor(w / math.sqrt(math.pi), dtype=torch.float32)
    return nodes, weights


class SavingPolicy(nn.Module):
    """Feasible by construction: k' = resources(k, z) * sigmoid(g_theta(normalized state))."""

    def __init__(self, rung: Rung) -> None:
        super().__init__()
        self.rung = rung
        lo, hi = (math.log(m * rung.k_ss) for m in DOMAIN)
        self.log_center, self.log_scale = (lo + hi) / 2.0, (hi - lo) / 2.0
        n_in = 2 if rung.stochastic else 1
        self.net = nn.Sequential(
            nn.Linear(n_in, rung.hidden), nn.Tanh(),
            nn.Linear(rung.hidden, rung.hidden), nn.Tanh(),
            nn.Linear(rung.hidden, 1),
        )
        if rung.informed_start:
            # Version 0 is the one state where the answer is known: at k_ss the household saves
            # k_ss out of resources(k_ss).  Start the share there, nearly flat, and let training bend it.
            share = rung.k_ss / (rung.productivity * rung.k_ss**rung.alpha + (1.0 - rung.delta) * rung.k_ss)
            with torch.no_grad():
                self.net[-1].bias.fill_(math.log(share / (1.0 - share)))
                self.net[-1].weight.mul_(0.1)

    def output(self, k: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        return z * self.rung.productivity * k.pow(self.rung.alpha)

    def resources(self, k: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        return self.output(k, z) + (1.0 - self.rung.delta) * k

    def saving_share(self, k: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        x = (torch.log(k) - self.log_center) / self.log_scale
        if self.rung.stochastic:
            x = torch.cat([x, torch.log(z) / self.rung.log_z_bound], dim=1)
        return torch.sigmoid(self.net(x))

    def forward(self, k: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        return self.resources(k, z) * self.saving_share(k, z)


def expected_rhs(model: SavingPolicy, k: torch.Tensor, z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Return c today and beta * E[ u'(c') * (1 + f'(k') - delta) ] under the model's policy."""
    r = model.rung
    k_next = model(k, z)
    c = model.resources(k, z) - k_next
    if r.stochastic:
        nodes, weights = gauss_hermite()
        z_next = torch.exp(r.rho * torch.log(z) + r.sigma * nodes.reshape(1, -1))  # (B, J)
    else:
        weights = torch.ones(1)
        z_next = torch.ones_like(k)
    k_rep = k_next.expand(-1, z_next.shape[1])
    flat_k, flat_z = k_rep.reshape(-1, 1), z_next.reshape(-1, 1)
    c_next = model.resources(flat_k, flat_z) - model(flat_k, flat_z)
    gross = r.alpha * flat_z * r.productivity * flat_k.pow(r.alpha - 1.0) + 1.0 - r.delta
    integrand = (c_next.pow(-r.gamma) * gross).reshape(z_next.shape)
    return c, r.beta * (integrand * weights.reshape(1, -1)).sum(dim=1, keepdim=True)


def log_residual(model: SavingPolicy, k: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    """log( beta E[u'(c') R'] / u'(c) ).  Unit-free; zero at the solution."""
    c, rhs = expected_rhs(model, k, z)
    return torch.log(rhs) + model.rung.gamma * torch.log(c)


def euler_error(model: SavingPolicy, k: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    """Judd's consumption-equivalent error |c_implied / c - 1|; report its log10."""
    c, rhs = expected_rhs(model, k, z)
    return (rhs.pow(-1.0 / model.rung.gamma) / c - 1.0).abs()


def sample_states(rung: Rung, n: int, band: tuple[float, float] = DOMAIN,
                  generator: torch.Generator | None = None) -> tuple[torch.Tensor, torch.Tensor]:
    lo, hi = (math.log(m * rung.k_ss) for m in band)
    k = torch.exp(lo + (hi - lo) * torch.rand(n, 1, generator=generator))
    if rung.stochastic:
        b = rung.log_z_bound * (band[1] / DOMAIN[1] if band != DOMAIN else 1.0)
        z = torch.exp(-b + 2.0 * b * torch.rand(n, 1, generator=generator))
    else:
        z = torch.ones(n, 1)
    return k, z


def train(rung_id: int, steps: int | None = None, informed_start: bool | None = None) -> tuple[SavingPolicy, list[tuple[int, float]], float]:
    """Train on Euler residuals only."""
    rung = RUNGS[rung_id]
    if informed_start is not None:
        rung = replace(rung, informed_start=informed_start)
    torch.manual_seed(rung.seed)
    torch.set_num_threads(1)
    model = SavingPolicy(rung)
    optimizer = torch.optim.Adam(model.parameters(), lr=rung.learning_rate)
    steps = rung.steps if steps is None else steps
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps + 1, eta_min=rung.learning_rate / 30)
    history: list[tuple[int, float]] = []
    started = time.perf_counter()
    for step in range(steps + 1):
        k, z = sample_states(rung, rung.batch_size)        # fresh states every step
        loss = log_residual(model, k, z).square().mean()   # the label is zero
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        scheduler.step()
        if step % max(steps // 20, 1) == 0 or step == steps:
            history.append((step, float(loss.detach())))
    model.eval()
    return model, history, time.perf_counter() - started


# ---------------------------------------------------------------- benchmarks

def closed_form(rung: Rung, k: np.ndarray, z: np.ndarray) -> np.ndarray:
    """Exact policy when utility is log and delta = 1."""
    if not (rung.gamma == 1.0 and rung.delta == 1.0):
        raise ValueError("closed form needs log utility and full depreciation")
    return rung.alpha * rung.beta * z * rung.productivity * k**rung.alpha


def time_iteration(rung: Rung, n_grid: int = 400, tol: float = 1e-10, max_iter: int = 2000) -> dict[str, object]:
    """Policy-function iteration on the deterministic Euler equation.  No value function.

    Given next period's policy g, today's k' solves u'(c) = beta u'(c'(k')) R(k') at each grid
    point; the root is bracketed and found by bisection, and g is interpolated in log k.
    """
    if rung.stochastic:
        raise ValueError("time iteration here is deterministic")
    a, b, d, gam, A = rung.alpha, rung.beta, rung.delta, rung.gamma, rung.productivity
    grid = np.exp(np.linspace(math.log(OFF_DOMAIN[0] * 0.8 * rung.k_ss),
                              math.log(OFF_DOMAIN[1] * 1.2 * rung.k_ss), n_grid))
    res = A * grid**a + (1 - d) * grid
    g = 0.5 * res if d == 1.0 else grid * 0.0 + np.minimum(grid, 0.95 * res)

    def gap(kp: np.ndarray, g_now: np.ndarray) -> np.ndarray:
        c = res - kp
        kpp = np.interp(np.log(kp), np.log(grid), g_now)
        c_next = np.maximum(A * kp**a + (1 - d) * kp - kpp, 1e-12)
        return c**(-gam) - b * c_next**(-gam) * (a * A * kp**(a - 1) + 1 - d)

    for it in range(max_iter):
        lo, hi = np.full(n_grid, 1e-8), res - 1e-10
        for _ in range(80):  # gap is increasing in k'
            mid = 0.5 * (lo + hi)
            positive = gap(mid, g) > 0
            hi, lo = np.where(positive, mid, hi), np.where(positive, lo, mid)
        g_new = 0.5 * (lo + hi)
        change = float(np.max(np.abs(g_new - g) / g))
        g = g_new
        if change < tol:
            break
    return {"grid": grid, "policy": g, "iterations": it + 1, "sup_change": change}


def benchmark_policy(rung_id: int, k: np.ndarray, z: np.ndarray, cache: dict | None = None) -> np.ndarray:
    rung = RUNGS[rung_id]
    if rung_id == 1:
        ti = cache if cache is not None else time_iteration(rung)
        return np.interp(np.log(k), np.log(ti["grid"]), ti["policy"])
    return closed_form(rung, k, z)


# ---------------------------------------------------------------- validation

def simulate_to_rest(model: SavingPolicy, periods: int = 3000) -> float:
    """Deterministic path (z = 1) from 0.5 k_ss; returns the terminal capital stock."""
    k = torch.tensor([[0.5 * model.rung.k_ss]])
    z = torch.ones(1, 1)
    with torch.no_grad():
        for _ in range(periods):
            k = model(k, z)
    return float(k)


def validate(rung_id: int, model: SavingPolicy, n: int = 4000, ti_cache: dict | None = None) -> dict[str, float]:
    """Held-out and off-domain diagnostics; states come from a generator training never used."""
    rung = model.rung
    gen = torch.Generator().manual_seed(777)
    out: dict[str, float] = {}
    with torch.no_grad():
        for label, band in (("held_out", DOMAIN), ("off_domain", OFF_DOMAIN)):
            k, z = sample_states(rung, n, band, gen)
            if label == "off_domain":  # keep only the states outside the training band
                keep = ((k < DOMAIN[0] * rung.k_ss) | (k > DOMAIN[1] * rung.k_ss)).squeeze()
                k, z = k[keep], z[keep]
            k_next = model(k, z)
            c = model.resources(k, z) - k_next
            err = euler_error(model, k, z)
            exact = benchmark_policy(rung_id, k.numpy(), z.numpy(), ti_cache)
            rel = np.abs(k_next.numpy() - exact) / exact
            out[f"{label}_feasible"] = bool(((c > 0) & (k_next > 0)).all())
            out[f"{label}_log10_max_euler_error"] = float(torch.log10(err.max()))
            out[f"{label}_log10_mean_euler_error"] = float(torch.log10(err.mean()))
            out[f"{label}_max_relative_policy_error"] = float(rel.max())
            out[f"{label}_median_relative_policy_error"] = float(np.median(rel))
    k_end = simulate_to_rest(model)
    out["k_ss_analytical"] = rung.k_ss
    out["k_ss_simulated"] = k_end
    out["k_ss_relative_gap"] = abs(k_end - rung.k_ss) / rung.k_ss
    return out


GATES = {
    "feasible": lambda v: v["held_out_feasible"] and v["off_domain_feasible"],
    "max policy error < 1% (held-out)": lambda v: v["held_out_max_relative_policy_error"] < 0.01,
    "steady state within 1%": lambda v: v["k_ss_relative_gap"] < 0.01,
}


def check_gates(v: dict[str, float]) -> dict[str, bool]:
    return {name: bool(test(v)) for name, test in GATES.items()}


# ---------------------------------------------------------------- fallbacks

def checkpoint_path(rung_id: int) -> Path:
    return DATA / f"growth_rung{rung_id}_reference.pt"


def load_reference(rung_id: int) -> tuple[SavingPolicy, dict[str, object]]:
    report = json.loads((DATA / "growth_ladder_report.json").read_text(encoding="utf-8"))
    entry = report["rungs"][str(rung_id)]
    if file_sha256(checkpoint_path(rung_id)) != entry["checkpoint_sha256"]:
        raise RuntimeError("growth checkpoint hash does not match its report")
    model = SavingPolicy(RUNGS[rung_id])
    model.load_state_dict(torch.load(checkpoint_path(rung_id), map_location="cpu", weights_only=True))
    model.eval()
    return model, entry


def build_references() -> dict[str, object]:
    """Train all rungs, validate, write checkpoints and the report.  Run once by the instructor."""
    report: dict[str, object] = {"helper": "s07_growth.py", "domain": DOMAIN,
                                 "off_domain": OFF_DOMAIN, "gauss_hermite_nodes": GH_NODES, "rungs": {}}
    ti = time_iteration(RUNGS[1])
    for rung_id, rung in RUNGS.items():
        model, history, seconds = train(rung_id)
        v = validate(rung_id, model, ti_cache=ti)
        torch.save(model.state_dict(), checkpoint_path(rung_id))
        entry = {"calibration": asdict(rung), "elapsed_seconds": seconds,
                 "final_training_loss": history[-1][1], "validation": v, "gates": check_gates(v),
                 "checkpoint_sha256": file_sha256(checkpoint_path(rung_id))}
        if rung_id == 1:
            cold, _, _ = train(1, informed_start=False)
            entry["uninformed_start"] = validate(1, cold, ti_cache=ti)
            entry["time_iteration"] = {"iterations": ti["iterations"], "sup_change": ti["sup_change"],
                                       "grid_points": len(ti["grid"])}
        report["rungs"][str(rung_id)] = entry
    (DATA / "growth_ladder_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report


if __name__ == "__main__":
    rep = build_references()
    for rid, e in rep["rungs"].items():
        print(rid, f"{e['elapsed_seconds']:.1f}s", e["gates"])
        for key, val in e["validation"].items():
            print("   ", key, val)

# Block A — PyTorch in five ideas

## A1. A tensor is an array that remembers its shape and its type

Most first bugs are shape bugs. A batch of 256 observations of one state variable is `(256, 1)`, not `(256,)`.

In [ ]:
print(broadcasting_demo())     # 7.B1: (4,1) minus (4,) is silently a (4,4) table

## A2. Autograd records the computation, then walks it backwards

The two worked examples of 7.A3, by hand **and** with `backward()`. The hand numbers and the autograd numbers
must agree to machine precision; this cell is the test behind every number on those slides.

In [ ]:
s = backprop_sigmoid()      # x = 1, t = 0, w1 = 0.5, b1 = 0, w2 = 2, b2 = 0, L = 0.5 (yhat - t)^2
for name in ("h", "z2", "yhat", "loss", "dyhat_dz2", "delta2", "dL_dw2", "dL_dh", "dh_dz1", "delta1",
             "dL_dw1", "w2_new", "w1_new"):
    print(f"{name:9s} {s[name]: .4f}")
assert close(s["autograd_dL_dw1"], s["dL_dw1"], 1e-12) and close(s["autograd_dL_dw2"], s["dL_dw2"], 1e-12)
assert close(s["dL_dw1"], LEDGER["backprop_sigmoid_dL_dw1"], 1e-6)
print("autograd agrees, and so does the slide")

r = backprop_relu()         # x = 2, y = 3, (w1, b1, v, c) = (0.5, -0.1, 1.2, 0.3), L = (yhat - y)^2
print(f"\nforward : z = {r['z']:.4f}, h = {r['h']:.4f}, yhat = {r['yhat']:.4f}, L = {r['loss']:.4f}")
print(f"backward: dL/dv = {r['dL_dv']:.4f}, dL/dc = {r['dL_dc']:.4f}, dL/dw1 = {r['dL_dw1']:.4f}, dL/db1 = {r['dL_db1']:.4f}")
print(f"update  : w1 -> {r['w1_new']:.4f}, v -> {r['v_new']:.4f};  loss {r['loss']:.4f} -> {r['loss_new']:.4f}")
print(f"dead unit: z = {r['dead_z']:.1f} < 0, so its weight's gradient is {r['dead_grad']:.1f} -- no signal, no learning")
assert close(r["dL_dw1"], LEDGER["backprop_relu_dL_dw1"]) and close(r["loss_new"], LEDGER["backprop_relu_loss_new"])

**Exercise A2.** Change the sigmoid example's target to $t=1$ and predict, before running, the sign of
$\partial L/\partial w_2$. Then add a third layer of sigmoids and watch $|\partial L/\partial w_1|$: 7.A2's
$0.25^L$ in one number.

## A3. The five-line loop, at version 0 --- where the answer is known

A linear "network" trained on squared error must land on the OLS coefficients (7.B1). If it does not, the bug
is in the loop, and you know before touching a real problem.

In [ ]:
v0 = version0()
print("OLS    :", np.round(v0["ols"], 4)); print("learned:", np.round(v0["learned"], 4))
assert v0["max_gap"] < 1e-3 and np.allclose(np.round(v0["ols"], 4), LEDGER["version0_ols"])

## A4. A network is a class; seeds make runs repeatable

In [ ]:
print(seed_check())          # 97 parameters; the same seed gives the same initial weights
X = torch.randn(64, 1); y = torch.sin(X)
print("one Adam step, train and validation loss:", one_training_step(nn.MSELoss(), X[:48], y[:48], X[48:], y[48:]))

**Exercise A4.** Delete `optimizer.zero_grad()` from a copy of `fit_curve_on` and fit the curve of Block B. No error
is raised. What happens to the loss, and why (7.B1)?

# Block B — Approximation, and which fit training picks

## B1. A curve whose answer is known

A hidden law $c = \ln w + 0.2\,w + \varepsilon$ on $w\in[1,11]$. The network is never told the functional form.
We hold out a test set, and then ask the fitted network about a $w$ it has never seen.

In [ ]:
w, c, tr, te, mu, sd = curve_data()
net = fit_curve_on(w, c, tr, mu, sd)
mse = score_curve(net, w, c, tr, te, mu, sd)
print({k: round(v, 4) for k, v in mse.items()}, "| noise variance = 0.0225")
assert close(mse["test"], LEDGER["curve_mse"][1], 1e-3)

with torch.no_grad():
    grid = torch.linspace(0.2, 30, 300).reshape(-1, 1)
    fitted = net((grid - mu) / sd)
    resid = (c[te] - net((w[te] - mu) / sd)).squeeze()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
inside = (grid >= 1) & (grid <= 11)
ax[0].scatter(w[tr], c[tr], s=6, color="0.7"); ax[0].plot(grid[inside], fitted[inside], color="C3")
ax[0].plot(grid[inside], torch.log(grid[inside]) + 0.2 * grid[inside], "k--", lw=1); ax[0].set_title("fit, inside the data")
ax[1].scatter(w[te], resid, s=8); ax[1].axhline(0, color="k", lw=0.8); ax[1].set_title("test residuals: no pattern left")
ax[2].plot(grid, fitted, color="C3", label="network"); ax[2].plot(grid, torch.log(grid) + 0.2 * grid, "k--", lw=1, label="truth")
ax[2].axvspan(1, 11, color="0.92"); ax[2].set_title("outside the grey band it is a straight line"); ax[2].legend()
plt.tight_layout(); plt.show()
with torch.no_grad():
    print(f"at w = 30: truth {math.log(30) + 6:.2f}, network {net((torch.tensor([[30.0]]) - mu) / sd).item():.2f}")

**Exercise B1.** Fit with `hidden=2`, then `hidden=256`; then `act=nn.Tanh`. A ReLU network is piecewise linear,
so outside the data it continues its last segment for ever; a tanh network flattens instead. Neither knows the law.

## B2. Kinks: fixed, then learned (7.A2)

One hidden ReLU layer is $a_0+\sum_i a_i\max(0,x-b_i)$. Fix the kinks $b_i$ and it is least squares; train them
and the approximant chooses where to bend.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for axis, H in zip(ax, (2, 4, 8, 32)):
    fit, kinks, m = kinks_learned(H)
    axis.plot(SIN_X, np.sin(SIN_X), "k--", lw=1); axis.plot(SIN_X, fit, color="C3")
    axis.plot(kinks, np.full_like(kinks, -1.25), "|", color="C0", ms=10)
    axis.set_title(f"H = {H}: MSE {m:.4f}, {len(kinks)} kinks in range")
    assert close(m, LEDGER["kinks_learned_mse"][H], 1e-4)
plt.tight_layout(); plt.show()
for H in (4, 8):
    print(f"{H} kinks: fixed by least squares {kinks_fixed(H)[2]:.4f}   learned {LEDGER['kinks_learned_mse'][H]:.4f}")

xg = np.linspace(-3.0, 3.0, 241); tgt = lambda z: np.tanh(8.0 * z)
_, _, fixed_mse = kinks_fixed(8, xg, tgt)
_, learned_kinks, learned_mse = kinks_learned(8, xg, tgt, steps=6000, seed=1)
print(f"\ntanh(8x): fixed {fixed_mse:.4f}, learned {learned_mse:.5f}, ratio {fixed_mse / learned_mse:.0f}")
print("learned kinks:", np.round(learned_kinks, 3))
assert close(fixed_mse / learned_mse, LEDGER["adaptivity_ratio"], 1e-2)

**Exercise B2.** On $\sin x$ with 8 units, least squares on fixed kinks beats training. Run `kinks_learned(8)` with
ten seeds and report the best and worst MSE. What does that say about 7.A2's "optimization error"?

## B3. Implicit regularization --- the precise statement (7.B4)

With more weights than data there are infinitely many exact fits. For a model **linear in its weights**, gradient
descent started at zero converges to the exact fit with the **smallest Euclidean norm**. That is a statement about
$\lVert a\rVert$, not about smoothness, and for a deep network it is a guide to intuition, not a theorem.

In [ ]:
xd, yd, shifts, a_min, a_other = minimum_norm_example()
Phi = hinge_features(xd, shifts)                          # 7 equations, 41 unknowns
a_gd = np.zeros(41)
step = 1.0 / np.linalg.norm(Phi, 2) ** 2                  # 1 / largest eigenvalue of Phi'Phi: the stable step
for _ in range(100000):
    a_gd -= step * Phi.T @ (Phi @ a_gd - yd)              # gradient descent from zero
for name, a in (("min-norm", a_min), ("GD from zero", a_gd), ("another exact fit", a_other)):
    print(f"{name:18s} max |fit error| = {np.abs(Phi @ a - yd).max():.1e}   ||a|| = {np.linalg.norm(a):.3f}")
print("||a_gd - a_min|| =", np.linalg.norm(a_gd - a_min).round(6))
assert close(np.linalg.norm(a_min), LEDGER["minnorm"][0]) and close(np.linalg.norm(a_other), LEDGER["minnorm"][1])
xg = np.linspace(-np.pi, np.pi, 300)
plt.figure(figsize=(6, 3.2))
plt.plot(xg, hinge_features(xg, shifts) @ a_other, color="0.6", label="another exact fit")
plt.plot(xg, hinge_features(xg, shifts) @ a_gd, color="C3", label="GD from zero")
plt.plot(xd, yd, "ko", ms=4); plt.legend(); plt.title("same data, same zero training error"); plt.show()

**Two levers: more parameters, or more data.** The hinge locations are taken in a *nested* order, so the
$p$-feature model is the $(p+1)$-feature model with one coefficient held at zero: every exact fit available at
$p$ is still available at $p+1$, so the minimum norm can only fall. Data points are nested too, so each new point
only adds a constraint, and the minimum can only rise.

In [ ]:
print("lever 1 -- more parameters, n = 7 fixed")
print(f"   {'p':>5} {'||a||':>8} {'test':>8}")
for pp in [2, 4, 6, 7, 8, 12, 24, 41, 64, 128, 256]:
    nm, te_, _, _ = min_norm_solve(7, pp)
    print(f"   {pp:5d} {nm:8.3f} {te_:8.4f}")
assert close(min_norm_solve(7, LEDGER["norm_peak"][0])[0], LEDGER["norm_peak"][1])

print("\nlever 2 -- more data, p = 41 fixed, nested points")
rows = []
for n_obs in range(3, 42):
    nm, te_, er, _ = min_norm_solve(n_obs, 41, nested_x=True)
    if er > 1e-9:
        break
    rows.append((n_obs, nm, te_))
for n_obs, nm, te_ in rows:
    if n_obs in (3, 5, 9, 17, 25, 37):
        print(f"   {n_obs:5d} {nm:8.3f} {te_:8.4f}")
assert close(rows[0][1], LEDGER["norm_vary_n"][0]) and close(rows[-1][1], LEDGER["norm_vary_n"][1])

More parameters drive $\lVert a\rVert$ down and the test error does not follow; more data drive it up and the test
error falls sharply. **The norm is not a measure of quality.**

**Is that just overfitting?** Run the sweep on clean and on noisy data, and watch the *training* error.

In [ ]:
rows = {p: r for p, *r in overfit_vs_selection()}
print(f"  {'p':>5} | {'clean train':>12} {'clean test':>11} | {'noisy train':>12} {'noisy test':>11}")
for pp in [2, 6, 7, 8, 12, 41, 256]:
    te0, te1, tr0, tr1 = rows[pp]
    print(f"  {pp:5d} | {tr0:12.1e} {te0:11.4f} | {tr1:12.1e} {te1:11.4f}")
assert close(rows[7][0], LEDGER["overfit_clean_noisy_at_n"][0]) and close(rows[7][1], LEDGER["overfit_clean_noisy_at_n"][1])

The training error is **exactly zero for every $p\ge n$ in both columns**, so 7.A3's diagnostic --- a training loss
below the noise floor --- cannot tell these fits apart. And the clean column has nothing to overfit, yet $p=n$ is
still a disaster. Overfitting is a criterion that is too permissive; this is a tie the criterion cannot break.

**Exercise B3.** Add noise to the data in lever 2 and re-run: does more data still lower the test error, and what
happens to the norm?

# Block C — Forecasting the 10-year Treasury yield one month ahead (7.B1)

**The contract.**

- **Target:** the month-end 10-year yield (`DGS10`) at $t+1$. Month-end, not the monthly average: averaging daily
  rates manufactures predictability in changes.
- **Information set, identical for every model:** the yield and the 10y--3m spread at month-end $t$; the federal
  funds rate, unemployment and 12-month CPI inflation for month $t-1$, because that is what has been published by
  the end of month $t$. The panel is current-vintage data, not what a forecaster saw in real time.
- **Splits by date, never shuffled:** train 1990--2014 · validate 2015--2019 (early stopping and width only) ·
  test 2020 onward. Scalers come from the training window.
- **Said in advance:** a random walk is very hard to beat for a long yield one month ahead, and the test window ---
  a pandemic, then the fastest tightening in forty years --- lies partly outside the training support.
  If the network loses, that is the lesson, not a failure of the lab.

In [ ]:
panel = load_panel()                           # verifies the file against its SHA-256 manifest
manifest = json.loads(MANIFEST.read_text())
print("downloaded", manifest["download_date"], "|", manifest["vintage"], "| rows:", len(panel))
parts = split(panel)
for name, frame in parts.items():
    print(f"{name:6s} labels dated {frame['target_date'].min().date()} -> {frame['target_date'].max().date()}  ({len(frame)})")
panel.tail(3)

In [ ]:
result = run_all(seeds=range(10))
table = pd.DataFrame(result["table"]).T[["rmse", "mae", "rmse_min", "rmse_max"]]
print("validation MSE by width (the only place width is chosen):", {k: round(v, 4) for k, v in result["width_scores"].items()})
print("\nTest window, percentage points.  MLP row: mean over 10 seeds, with the range of RMSE.")
print(table.round(4).to_string(na_rep=""))
ens = metrics(result["actual"], result["predictions"]["mlp_seed_mean"])
print(f"\naverage of the ten forecasts: RMSE {ens['rmse']:.4f}   (random walk {result['table']['random_walk']['rmse']:.4f})")
for k, v in LEDGER["yield_rmse"].items():
    assert close(result["table"][k]["rmse"], v, 1e-3), k
assert close(ens["rmse"], LEDGER["yield_ensemble_rmse"], 1e-3)

In [ ]:
support = pd.DataFrame(result["support"], index=["train min", "train max", "share of test months outside"]).T
print(support.round(2).to_string())

test = parts["test"]
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].plot(panel["target_date"], panel["y10_next"], color="0.3")
for (lo, hi), shade in (((TRAIN[0], TRAIN[1]), "0.93"), ((VALID[0], VALID[1]), "0.85")):
    ax[0].axvspan(pd.Timestamp(lo), pd.Timestamp(hi), color=shade)
ax[0].set_title("10-year yield: train (light), validate (dark), test (white)")
err_rw = result["actual"] - result["predictions"]["random_walk"]
err_nn = result["actual"] - result["predictions"]["mlp_seed_mean"]
ax[1].plot(test["target_date"], np.abs(err_rw), label="random walk", color="0.4")
ax[1].plot(test["target_date"], np.abs(err_nn), label="MLP (seed mean)", color="C3", alpha=0.8)
ax[1].set_title("absolute forecast error, test window"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# The hand-off to Session 9: dated forecasts, frozen.  A text signal dated <= t will be merged on
# `forecast_made` and scored against these numbers, unchanged.  Read, never rewritten here.
baseline = pd.read_csv(BASELINE)
print(baseline.head(3).round(3).to_string(index=False))
print("largest gap to this run's averaged forecast:",
      float(np.max(np.abs(baseline["mlp_seed_mean"].to_numpy() - result["predictions"]["mlp_seed_mean"]))))

**Exercise C.** (1) Shuffle before splitting and watch the test RMSE "improve" --- then explain why that number is
worthless (7.A3, leakage). (2) Fit the scaler on the full sample: how much moves? (3) Replace the target by the
*monthly-average* yield (average the daily file within the month) and see how much easier the "forecast" becomes.

# Block D — Solving the growth model as supervised learning (7.B2)

| Supervised learning | Here |
|---|---|
| inputs $x_i$ | sampled states $k_i$ (and $z_i$) |
| model $f_\theta$ | the saving policy $k' = \text{resources}(k,z)\cdot\sigma(g_\theta(k,z))$ --- feasible by construction |
| prediction error | the unit-free log Euler residual $\log\big(\beta\,E[u'(c')R'] / u'(c)\big)$ |
| label | **zero** --- supplied by the Euler equation, not by data |

No benchmark policy ever enters a loss. Three rungs, one code path (`s07_growth.py`, inlined above).

## D0. Rung 0 --- the closed form, and the reveal

Log utility (CRRA with $\gamma=1$), $\delta=1$, $\alpha=0.33$, $\beta=0.96$, $A=1$: the policy is $k'=\alpha\beta Ak^\alpha$.
Then look at what the network actually had to learn.

In [ ]:
ti = time_iteration(RUNGS[1])          # rung 1's benchmark (3.A3); computed once, used below
print(f"time iteration: {ti['iterations']} rounds on {len(ti['grid'])} nodes")
m0, hist0, sec0 = train(0)
v0 = validate(0, m0)
print(f"trained in {sec0:.1f}s | gates:", check_gates(v0))
with torch.no_grad():
    kk = torch.linspace(0.2, 2.0, 7).reshape(-1, 1) * RUNGS[0].k_ss
    print("learned saving share across k:", m0.saving_share(kk, torch.ones_like(kk)).squeeze().numpy().round(4))
print("alpha * beta =", 0.33 * 0.96, " <- the target was a single number; a bias term could pass this test")
assert ti["iterations"] == LEDGER["ti_iterations"]

## D1. Rung 1 --- a real function to approximate, and why a small residual is not enough

CRRA $\gamma=2$, $\delta=0.025$, $\alpha=0.33$, $\beta=0.99$ (quarterly). The saving share now varies with $k$.
The benchmark is **time iteration on the Euler equation**. Train twice: from PyTorch's default start, and from a
start where the share equals its known steady-state value.

In [ ]:
m1_cold, hist_cold, _ = train(1, informed_start=False)
m1, hist1, sec1 = train(1)
v_cold, v1 = validate(1, m1_cold, ti_cache=ti), validate(1, m1, ti_cache=ti)
show = ["held_out_log10_max_euler_error", "held_out_max_relative_policy_error", "k_ss_simulated", "k_ss_analytical"]
print(pd.DataFrame({"default start": v_cold, "steady-state start": v1}).loc[show].round(4).to_string())
print("gates, default start     :", check_gates(v_cold))
print("gates, steady-state start:", check_gates(v1))

r1 = RUNGS[1]
kg = torch.exp(torch.linspace(math.log(0.2 * r1.k_ss), math.log(2 * r1.k_ss), 200)).reshape(-1, 1); one = torch.ones_like(kg)
bench = benchmark_policy(1, kg.numpy(), one.numpy(), ti).ravel()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
with torch.no_grad():
    for model, label, color in ((m1_cold, "default start", "0.55"), (m1, "steady-state start", "C3")):
        ax[0].plot(kg, model(kg, one) - kg, color=color, label=label)
        ax[1].semilogy(kg, euler_error(model, kg, one), color=color, label=label)
ax[0].plot(kg, bench - kg.numpy().ravel(), "k--", lw=1, label="time iteration"); ax[0].axhline(0, color="k", lw=0.6)
ax[0].set_title("net saving k' - k"); ax[0].legend(); ax[1].set_title("Euler equation error (consumption units)"); ax[1].legend()
plt.tight_layout(); plt.show()

Read the table before the picture. The default start reaches an Euler error near $10^{-2}$ --- which sounds
acceptable --- with a policy that is wrong by a quarter and a simulated economy that runs down to nothing.
With $\beta R\approx1$ each quarter, the residual is nearly flat in the policy: **the Euler equation is necessary,
not sufficient**, and a residual is a local statement. What caught the error was the independent benchmark and
the steady-state check, not the loss.

## D2. Rung 2 --- an expectation inside the residual

Brock--Mirman: log utility, $\delta=1$, $\log z' = \rho\log z + \sigma\varepsilon'$ with $\rho=0.9$, $\sigma=0.05$.
The exact policy is $k'=\alpha\beta zAk^{\alpha}$. The expectation is computed with seven Gauss--Hermite nodes (4.B2):
deterministic, so the loss stays differentiable and carries no sampling noise.

First, why not simply draw shocks? The loss squares the residual, and **the square of a Monte Carlo mean is a
biased estimate of the squared mean**: $E[\bar X^2] = (E X)^2 + \mathrm{Var}(X)/n$.

In [ ]:
rng = np.random.default_rng(0)
h = lambda e: np.exp(0.05 * e) - math.exp(0.05**2 / 2)        # E[h] = 0 exactly, so the true squared mean is 0
for n in (1, 10, 100):
    draws = h(rng.standard_normal((200000, n))).mean(axis=1)
    print(f"n = {n:3d} draws: average of (MC mean)^2 = {np.mean(draws**2):.2e}   Var/n = {h(rng.standard_normal(10**6)).var() / n:.2e}")
x, wgt = gauss_hermite()
print(f"Gauss-Hermite, 7 nodes: (quadrature mean)^2 = {float((wgt * (torch.exp(0.05 * x) - math.exp(0.05**2 / 2))).sum())**2:.1e}")

m2, hist2, sec2 = train(2)
v2 = validate(2, m2)
print(f"\ntrained in {sec2:.1f}s | gates:", check_gates(v2))

## D3. Validation: held-out, off-domain, benchmark, steady state

Training states were drawn from $[0.2,\,2]\times k^{*}$. *Held-out* states are fresh draws from that band;
*off-domain* states lie in $[0.1,0.2)\cup(2,3]\times k^{*}$, where the network was never asked anything.
Errors are on Judd's scale: $\log_{10}$ of the consumption-equivalent Euler error ($-3$ = one dollar in a thousand).

In [ ]:
rows = ["held_out_feasible", "off_domain_feasible", "held_out_log10_max_euler_error", "off_domain_log10_max_euler_error",
        "held_out_max_relative_policy_error", "off_domain_max_relative_policy_error", "k_ss_relative_gap"]
summary = pd.DataFrame({"rung 0": v0, "rung 1": v1, "rung 2": v2}).loc[rows]
print(summary.to_string(float_format=lambda f: f"{f:.4f}"))
gates = {f"rung {i}": check_gates(v) for i, v in enumerate((v0, v1, v2))}
print(pd.DataFrame(gates).to_string())
assert all(all(g.values()) for g in gates.values()), "a gate failed: load the reference checkpoints below"
for i, v in enumerate((v0, v1, v2)):
    print(f"rung {i}: held-out {v['held_out_log10_max_euler_error']:.3f} (slide {LEDGER['held_out_log10_max_euler'][str(i)]:.3f}),"
          f" off-domain {v['off_domain_log10_max_euler_error']:.3f} (slide {LEDGER['off_domain_log10_max_euler'][str(i)]:.3f})")

In [ ]:
# Fallback: validated reference policies (the AI-ECON recorded run), verified against their hashes,
# held to the same gates.
for rung_id in (0, 1, 2):
    ref, entry = load_reference(rung_id)
    print(rung_id, check_gates(validate(rung_id, ref, ti_cache=ti)))

**Exercise D.** (1) On rung 1, train the default start for ten times as many steps --- does the residual or the
policy error improve first? (2) Replace Gauss--Hermite by two Monte Carlo draws per state on rung 2 and compare the
gates. (3) Widen the training band to $[0.1, 3]$: what happens to the off-domain column, and what is "off-domain" now?

**What Session 7 did not do.** No value function was approximated anywhere above: the benchmark on rung 1 iterates
on the policy, and the network is a policy. Value functions, Bellman targets and learning by acting are Session 8.

# Appendix 1 --- A text classifier, and what its test accuracy measures (7.A3)

Sentences are built as *subject × phrase*. A random split puts every phrase in both training and test, so test
accuracy measures **recall of a template already seen**. Splitting **by phrase** asks the honest question.

In [ ]:
lk = text_leakage()
print(f"random split          : {lk['random_split']:.2f}   <- template recall")
print(f"split by phrase (6 | 2): {lk['by_phrase']:.2f}   <- phrases never seen in training")
assert (round(lk["random_split"], 2), round(lk["by_phrase"], 2)) == tuple(LEDGER["leakage"])

# Appendix 2 --- One input, four architectures (7.B4)

7.B3 says an architecture is a **constraint on $W$**. Here is that claim as arithmetic: one input $x=(1,2,3,4)$,
four readings of it, weights chosen rather than trained so every number can be checked by hand. Then break each
assumption in turn.

In [ ]:
fa = four_architectures()          # every value asserted inside the function
print("W of the convolution (three free numbers, the rest zero or tied):")
print(np.array(fa["conv"]["W"]))
for name in ("dense", "conv", "rnn", "gnn"):
    out = fa[name].get("y", fa[name].get("h"))
    print(f"{name:6s} -> {out}   {fa[name]['params']} free numbers")
print("\nshuffle the input (1,2,3,4) -> (3,1,4,2):")
print("  convolution:", fa["input_shuffle"]["conv"], "  (was", fa["conv"]["y"], ")")
print("  dense, weights relabelled the same way:", fa["input_shuffle"]["dense"], "  (was", fa["dense"]["y"], ")")
print("relabel the graph's nodes A B C D as D C B A:", fa["gnn"]["permuted"], "  sum readout", fa["gnn"]["readout_sum"])

# Appendix 3 --- What a gradient costs (7.A3)

Backpropagation delivers the whole gradient for a small multiple of one forward pass, **whatever $P$ is**. Measure it.
Your times will differ from anyone else's; the *ratio* is the point.

In [ ]:
def cost(net, n, d_in, reps=20):
    X = torch.randn(n, d_in)
    P = sum(p.numel() for p in net.parameters())
    tgt = torch.randn(n, net(X[:1]).shape[1])
    loss_fn = nn.MSELoss()
    def run(backward):
        t0 = time.perf_counter()
        for _ in range(reps):
            out = net(X)
            if backward:
                net.zero_grad(); loss_fn(out, tgt).backward()
        return (time.perf_counter() - t0) / reps
    run(True)                                     # warm up
    fwd = min(run(False) for _ in range(3))
    both = min(run(True) for _ in range(3))
    return P, fwd, both

print(f"{'network':26s} {'P':>8s} {'forward':>10s} {'fwd+bwd':>10s} {'ratio':>7s}")
for name, net_, n, d in [
    ("digits 784-128-10", nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)), 60000, 784),
    ("yield MLP 5-4-1",   nn.Sequential(nn.Linear(5, 4), nn.ReLU(), nn.Linear(4, 1)), 300, 5),
    ("growth policy 1-32-32-1", nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1)), 256, 1),
]:
    P, fwd, both = cost(net_, n, d)
    print(f"{name:26s} {P:8d} {1e3*fwd:9.2f}ms {1e3*both:9.2f}ms {both/fwd:7.2f}")
print("\nFinite differences would need P + 1 forward passes for the same gradient: 101,771 for the digits network.")

# What to carry forward

1. **Check the machine where the answer is known**: `backward()` against the hand derivative, a linear network
   against OLS, rung 0 against the closed form.
2. **A test number is honest only if the test set did no other job**: split by date, scale on the training window,
   choose the design on validation data.
3. **A network chooses its own basis.** Where the curvature is concentrated that pays; where it is not, least
   squares on fixed kinks can win.
4. **With more parameters than data, the optimizer picks the fit** --- and a training loss of zero cannot tell you
   which one.
5. **On a month-ahead yield, nothing here beats a random walk.** Report it anyway.
6. **The Euler equation is necessary, not sufficient.** A small residual from a bad start is still a bad policy;
   the benchmark and the steady-state check caught it, the loss did not.
7. **Claim a solution only where you trained and tested it.** Off the training band, accuracy falls by up to two
   orders of magnitude.